# Código para estraer mensajes de meta

In [ ]:
import requests
import pandas as pd
import json

page_id = '460399690488537' # your page id, ex: '123456789'
post_id = '18317340889172833' # your post id, ex: '123456789'
access_token = 'EAAQ9FrwwOUABPZCnNH67QhE5cJvUDr9u0cMM2GE50zvsMPQu8ZCZBm85U7PGWfeXTsFjPyVZAwqsCZCD5oqjtQp5qA9SWLyZBtWq5Q1SvaZAQuohuGFS2KUn6sccDFwke2V0ZBQkZCUNSJIYyZCqPfpLa57mE6FZBIvsrTuLMiSmVx8UoxjN0o194CRg39ml6of' # your access token, from https://developers.facebook.com/tools/explorer/

url = f'https://graph.facebook.com/v24.0/{page_id}_{post_id}/comments?access_token={access_token}'

response = requests.request("GET", url)

# save name, time, message in excel file
data = json.loads(response.text)

# create object with only name, time, message
def get_comment(comment):
    return {
        'name': comment['from']['name'],
        'time': comment['created_time'],
        'message': comment['message']
    }

excel_data = list(map(get_comment, data['data']))
df = pd.DataFrame(excel_data)
df.to_excel('comments.xlsx', index=False)

KeyError: 'data'

In [7]:
data

{'error': {'message': "Unsupported get request. Object with ID '460399690488537_18317340889172833' does not exist, cannot be loaded due to missing permissions, or does not support this operation. Please read the Graph API documentation at https://developers.facebook.com/docs/graph-api",
  'type': 'GraphMethodException',
  'code': 100,
  'error_subcode': 33,
  'fbtrace_id': 'Afar5NX2IqUwJ45OmO10-1e'}}

In [ ]:
import requests
import json

# --- Configuración ---
POST_ID = "18317340889172833"  # P.ej., "123456789_987654321"
ACCESS_TOKEN = "EAAQ9FrwwOUABPZCnNH67QhE5cJvUDr9u0cMM2GE50zvsMPQu8ZCZBm85U7PGWfeXTsFjPyVZAwqsCZCD5oqjtQp5qA9SWLyZBtWq5Q1SvaZAQuohuGFS2KUn6sccDFwke2V0ZBQkZCUNSJIYyZCqPfpLa57mE6FZBIvsrTuLMiSmVx8UoxjN0o194CRg39ml6of"
BASE_URL = "https://graph.facebook.com/v24.0/" # Siempre usa una versión explícita

def descargar_comentarios_de_post(post_id, access_token):
    """
    Descarga todos los comentarios de un post específico,
    manejando la paginación.
    """
    todos_los_comentarios = []
    
    # Endpoint de la API para los comentarios del post
    # Pedimos campos específicos: id, mensaje, fecha de creación y de quién es.
    url = f"{BASE_URL}{post_id}/comments"
    params = {
        'access_token': access_token,
        'fields': 'id,message,created_time,from{id,name}',
        'limit': 100  # Pedir hasta 100 comentarios por página
    }

    print(f"Iniciando descarga de comentarios para el post: {post_id}")

    while url:
        try:
            response = requests.get(url, params=params)
            data = response.json()

            # Manejo de errores de la API
            if 'error' in data:
                print(f"Error de API: {data['error']['message']}")
                break

            # Guardar los comentarios de esta página
            if 'data' in data and data['data']:
                todos_los_comentarios.extend(data['data'])
                print(f"  > Descargados {len(data['data'])} comentarios. Total: {len(todos_los_comentarios)}")

            # Paginación: ¿Hay una página siguiente?
            if 'paging' in data and 'next' in data['paging']:
                # Facebook nos da la URL completa para la siguiente página
                url = data['paging']['next']
                # Limpiamos los params porque la URL 'next' ya los contiene
                params = {} 
            else:
                # No hay más páginas, terminamos el bucle
                url = None
                
        except requests.RequestException as e:
            print(f"Error de conexión: {e}")
            break

    print(f"Descarga finalizada. Total de comentarios: {len(todos_los_comentarios)}")
    return todos_los_comentarios

# --- Ejecución del script ---
if __name__ == "__main__":
    
    comentarios = descargar_comentarios_de_post(POST_ID, ACCESS_TOKEN)
    
    if comentarios:
        # Ahora tienes la lista 'comentarios' lista para tu análisis de sentimientos.
        # Por ejemplo, imprimamos el primero:
        print("\n--- Ejemplo de un comentario ---")
        print(json.dumps(comentarios[0], indent=2, ensure_ascii=False))

        # O guardar todo en un archivo JSON para procesar después
        try:
            with open('comentarios_facebook.json', 'w', encoding='utf-8') as f:
                json.dump(comentarios, f, indent=4, ensure_ascii=False)
            print("\n[+] Comentarios guardados en 'comentarios_facebook.json'")
        except IOError as e:
            print(f"Error al guardar el archivo: {e}")